# Baseline model

Predicting whether an unplanned New York hospital admission lasts more than
7 days. Decisions are in `docs/notes/2026-11-05-data-cohort-split.md` and
`docs/notes/2026-11-06-cleaning-encoding-baseline.md`.

In [1]:
import re
import urllib.request
import warnings
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

SEED = 42

## Step 1: Get the data

Download the 2023 and 2024 SPARCS files unless they're already there, then
stop with an error if either file isn't the expected size.

In [2]:
RAW_DIR = Path("data/raw")
URL = "https://health.data.ny.gov/api/views/{id}/rows.csv?accessType=DOWNLOAD"
DATASETS = {2023: "46xm-urtu", 2024: "sf4k-39ay"}
EXPECTED_ROWS = {2023: 2_125_754, 2024: 2_196_737}
EXPECTED_COLUMNS = 33

raw = {}
for year, dataset_id in DATASETS.items():
    path = RAW_DIR / f"sparcs_{year}.csv"
    if not path.exists():
        print(f"Downloading {year}...")
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(URL.format(id=dataset_id), path)

    raw[year] = pd.read_csv(path, dtype=str, engine="pyarrow")
    assert raw[year].shape == (EXPECTED_ROWS[year], EXPECTED_COLUMNS), (
        f"{path} has shape {raw[year].shape}. If the download was interrupted, "
        "delete the file and run this again."
    )
    print(f"{year}: {raw[year].shape[0]:,} rows, {raw[year].shape[1]} columns")

2023: 2,125,754 rows, 33 columns


2024: 2,196,737 rows, 33 columns


## Step 2: Build the cohort and sample

This step only lines up the columns, creates the target and chooses the rows.
Before each decision, we look at the data.

### Pool the two years

First, do the two files use the same column names?

In [3]:
def snake(name):
    return re.sub(r"[^0-9a-z]+", "_", name.lower()).strip("_")


columns = {year: {snake(c) for c in df.columns} for year, df in raw.items()}
print("Only in 2023:", sorted(columns[2023] - columns[2024]))
print("Only in 2024:", sorted(columns[2024] - columns[2023]))
print("Year column present:", "discharge_year" in columns[2023] & columns[2024])

Only in 2023: ['hospital_service_area', 'zip_code_3_digits']
Only in 2024: ['health_service_area', 'zip_code']
Year column present: True


Two columns have different names, so we rename them. Without this, each would
split into two half-empty columns. The files already have a year column, so we
don't need to add one.

In [4]:
RENAMES = {
    "hospital_service_area": "health_service_area",
    "zip_code_3_digits": "zip_code",
}

frames = [df.rename(columns=snake).rename(columns=RENAMES) for df in raw.values()]
assert set(frames[0].columns) == set(frames[1].columns)

cohort = pd.concat(frames, ignore_index=True)
del raw, frames
print(f"{len(cohort):,} stays")

4,322,491 stays


### Create the target

Length of stay is stored as text. Which values aren't plain whole numbers?

In [5]:
is_number = cohort["length_of_stay"].str.fullmatch(r"\d+")
print(cohort.loc[~is_number, "length_of_stay"].value_counts().to_string())

length_of_stay
120+     2404
120 +    2290


The only exception is the top value, spelled "120 +" in 2023 and "120+" in
2024. We treat both as 120. A long stay is more than 7 days, so $y = 1$ if
length of stay is at least 8.

In [6]:
los_text = cohort["length_of_stay"].str.replace(" ", "").replace("120+", "120")
cohort["los"] = pd.to_numeric(los_text)
cohort["y"] = (cohort["los"] >= 8).astype(int)

print(f"Length of stay ranges from {cohort['los'].min()} to {cohort['los'].max()}")
print(f"Long stays: {cohort['y'].mean():.1%} of all stays")

Length of stay ranges from 1 to 120
Long stays: 20.0% of all stays


In [7]:
def long_stay_table(df, by):
    """Number of stays and long-stay rate for each level of a column."""
    table = df.groupby(by, dropna=False)["y"].agg(stays="size", long_stay_rate="mean")
    return table.sort_values("stays", ascending=False).round(3)

### Admission type

Which kinds of admission are in the data?

In [8]:
print(long_stay_table(cohort, "type_of_admission").to_string())

                     stays  long_stay_rate
type_of_admission                         
Emergency          2870379           0.229
Elective            712698           0.143
Newborn             392468           0.054
Urgent              322676           0.242
Trauma               18117           0.274
Not Available         6153           0.349


We're predicting long stays for unplanned admissions, so we keep emergency,
urgent and trauma. Elective stays are largely planned, newborn stays work
differently, and "Not Available" doesn't tell us the type.

In [9]:
KEEP_ADMISSION_TYPES = ["Emergency", "Urgent", "Trauma"]

cohort = cohort[cohort["type_of_admission"].isin(KEEP_ADMISSION_TYPES)]
print(f"{len(cohort):,} stays remain")

3,211,172 stays remain


### Newborns

The admission type filter removes stays labeled "Newborn". Are any newborns
left under other admission types? APR MDC code 15 is the newborn diagnostic
group.

In [10]:
newborns = cohort[cohort["apr_mdc_code"] == "15"]
print(f"{len(newborns):,} stays, {newborns['y'].mean():.1%} long")
print()
print(long_stay_table(newborns, "type_of_admission").to_string())
print()
print("Top diagnoses:")
diagnoses = newborns["ccsr_diagnosis_description"].str.capitalize()
print(diagnoses.value_counts().head(5).to_string())

10,250 stays, 19.7% long

                   stays  long_stay_rate
type_of_admission                       
Emergency           6015           0.080
Urgent              4217           0.364
Trauma                18           0.167

Top diagnoses:
ccsr_diagnosis_description
Hemolytic jaundice and perinatal jaundice               3389
Liveborn                                                2730
Other specified and unspecified perinatal conditions     895
Respiratory distress syndrome                            589
Respiratory perinatal condition                          463


Some are, though they're a small group. Our plan leaves newborns out because
their stays work differently, so we drop this group too.

In [11]:
cohort = cohort[cohort["apr_mdc_code"] != "15"]
print(f"{len(cohort):,} stays remain")

3,200,922 stays remain


### Redacted records

There's no column flagging redacted records, but `facility_name` says
"Redacted" on some rows. What's blank on those rows?

In [12]:
LOCATION_COLUMNS = [
    "permanent_facility_id",
    "hospital_county",
    "health_service_area",
    "zip_code",
]

is_redacted = cohort["facility_name"].str.contains("Redacted", na=False)
print(
    f"{is_redacted.sum():,} redacted stays, {cohort.loc[is_redacted, 'y'].mean():.1%} long"
)
print(f"Blank facility ID: {cohort['permanent_facility_id'].isna().sum():,} stays")
print()
print("Share of rows blank:")
blank = cohort[LOCATION_COLUMNS].isna().groupby(is_redacted).mean().round(3)
print(blank.rename_axis("redacted").to_string())
print()
print("Top diagnostic groups in redacted stays:")
print(cohort.loc[is_redacted, "apr_mdc_code"].value_counts().head(3).to_string())

8,606 redacted stays, 7.8% long
Blank facility ID: 8,606 stays

Share of rows blank:


          permanent_facility_id  hospital_county  health_service_area  zip_code
redacted                                                                       
False                       0.0              0.0                  0.0     0.016
True                        1.0              1.0                  1.0     1.000

Top diagnostic groups in redacted stays:
apr_mdc_code
14    5872
19     396
04     318


The publisher blanks the location on these rows for privacy, and its
documentation describes this as extra de-identification for abortion records.
Every redacted row has no facility, county, service area or ZIP. A real hospital
always knows which facility it is, so these rows don't fit how the model would
be used, and we drop them.

This removes mostly pregnancy-related stays (MDC 14), so the paper needs to say
so.

In [13]:
cohort = cohort[cohort["permanent_facility_id"].notna()]
print(f"{len(cohort):,} stays remain")

3,192,316 stays remain


### Specialty facilities

We want a model for typical hospitals. The publisher lists 10 facilities whose
discharges are mostly rehab or life support. How many of their stays are left?

In [14]:
SPECIALTY_FACILITIES = [
    "000340",
    "000775",
    "000831",
    "000968",
    "001046",
    "001138",
    "001460",
    "001486",
    "009431",
    "010223",
]

is_specialty = cohort["permanent_facility_id"].isin(SPECIALTY_FACILITIES)
print(f"{is_specialty.sum():,} stays, {cohort.loc[is_specialty, 'y'].mean():.1%} long")
print()
print(long_stay_table(cohort[is_specialty], "permanent_facility_id").to_string())

176 stays, 59.7% long

                       stays  long_stay_rate
permanent_facility_id                       
001046                   106           0.915
000968                    63           0.063
000831                     4           0.750
001460                     2           0.000
000340                     1           1.000


Very few, and only five of the 10 facilities have any stays left. We drop them
so that the rule is simple: the publisher's list defines a specialty facility.

In [15]:
cohort = cohort[~is_specialty]
print(f"{len(cohort):,} stays remain")

3,192,140 stays remain


### Kept on purpose

These groups might look like candidates for removal, but we keep all of them.

In [16]:
died = cohort["patient_disposition"] == "Expired"
print(
    f"Patients who died: {died.sum():,} stays, {cohort.loc[died, 'y'].mean():.1%} long"
)
print(f"Exact duplicate rows: {cohort.duplicated().sum():,} extra rows")

is_blank_zip = cohort["zip_code"].isna()
print(f"Blank ZIP code: {is_blank_zip.sum():,} stays")
print()

is_behavioral = cohort["apr_mdc_code"].isin(["19", "20"])
facilities = cohort.assign(behavioral=is_behavioral).groupby("permanent_facility_id")
facilities = facilities.agg(
    stays=("y", "size"),
    long_stay_rate=("y", "mean"),
    behavioral_share=("behavioral", "mean"),
    name=("facility_name", "first"),
)
print("Highest long-stay rates among facilities with at least 200 stays:")
highest = facilities[facilities["stays"] >= 200]
highest = highest.sort_values("long_stay_rate", ascending=False).head(6).round(3)
print(highest.to_string())

Patients who died: 98,675 stays, 45.4% long


Exact duplicate rows: 184 extra rows
Blank ZIP code: 50,889 stays

Highest long-stay rates among facilities with at least 200 stays:
                       stays  long_stay_rate  behavioral_share                                                        name
permanent_facility_id                                                                                                     
000482                   434           0.836             0.355           St. Mary's Healthcare - Amsterdam Memorial Campus
001047                  4319           0.830             0.987  New York-Presbyterian Westchester Behavioral Health Center
001133                  4405           0.770             0.999          St. Joseph's MC-St. Vincent's Westchester Division
015437                  1215           0.725             0.991                     MOUNT SINAI -  BEHAVIORAL HEALTH CENTER
009059                   871           0.719             0.983                     Crouse Hospital - Commonwealth Division
000412

- **Patients who died:** death isn't known at admission, so removing them would
  use future information. Their stays are cut short, which the paper should say.
- **Exact duplicate rows:** with no patient ID, we can't tell whether these are
  errors or separate stays that look alike.
- **Blank ZIP codes:** ZIP is the patient's home area, not the hospital's
  location, so we handle these blanks during cleaning.
- **Facilities with many long stays:** these aren't on the publisher's list.
  `behavioral_share` is the share of stays for mental health or substance use
  (MDC 19 and 20). Most of these sites are nearly all behavioral, but not all of
  them. Choosing sites by long-stay rate would select on the outcome we're
  predicting, so they stay in and the paper notes it.

### Check the cohort

Stop with an error if the cohort isn't the expected size.

In [17]:
EXPECTED_COHORT = {"2023": 1_562_648, "2024": 1_629_492}

stays_by_year = cohort["discharge_year"].value_counts().to_dict()
assert stays_by_year == EXPECTED_COHORT, stays_by_year
assert len(cohort) == 3_192_140, len(cohort)
assert cohort["los"].notna().all()
assert round(cohort["y"].mean(), 4) == 0.2309, cohort["y"].mean()

print(f"Cohort: {len(cohort):,} stays, {cohort['y'].mean():.2%} long")
print(pd.Series(stays_by_year).to_string())

Cohort: 3,192,140 stays, 23.09% long
2024    1629492
2023    1562648


### Draw the sample

To keep the slower models practical to train, all models use one simple random
sample of 500,000 stays from the roughly 3.2 million in the cohort.

In [18]:
SAMPLE_SIZE = 500_000

sample = cohort.sample(n=SAMPLE_SIZE, random_state=SEED)
print(f"Sample: {len(sample):,} stays, {sample['y'].mean():.2%} long")

Sample: 500,000 stays, 23.06% long


Does the sample look like the cohort? The two columns below are the percentage
of stays at each level.

In [19]:
for column in ["discharge_year", "type_of_admission", "payment_typology_1", "race"]:
    shares = pd.DataFrame(
        {
            "cohort": cohort[column].value_counts(normalize=True, dropna=False),
            "sample": sample[column].value_counts(normalize=True, dropna=False),
        }
    )
    print((shares * 100).round(2).to_string())
    print()

                cohort  sample
discharge_year                
2024             51.05   51.08
2023             48.95   48.92

                   cohort  sample
type_of_admission                
Emergency           89.49   89.45
Urgent               9.94    9.97
Trauma               0.57    0.57

                           cohort  sample
payment_typology_1                       
Blue Cross/Blue Shield       7.56    7.57
Department of Corrections    0.09    0.08
Federal/State/Local/VA       0.99    1.02
Managed Care, Unspecified    1.21    1.19
Medicaid                    27.73   27.63
Medicare                    48.63   48.74
Miscellaneous/Other          1.00    0.99
Private Health Insurance    11.45   11.48
Self-Pay                     1.34    1.30
NaN                          0.00    0.00

                        cohort  sample
race                                  
White                    52.18   52.30
Other Race               26.68   26.53
Black/African American   19.83   19.87
Mult

## Step 3: Split

We split the sample into three sets, keeping the long-stay rate the same in each:

- **Train (360,000):** fitting, and K-fold CV for tuning. CV does the job of a
  validation set, so there isn't a separate one.
- **Recalibration (40,000):** fitting Platt scaling and isotonic regression only.
- **Test (100,000):** final scoring, once, on the same rows for every model. It
  stays untouched until the end, so the baseline checkpoint score comes from CV
  on the training set.

We take the test set off first, then take the recalibration set from what's left.

In [20]:
TRAIN_SIZE = 360_000
RECAL_SIZE = 40_000
TEST_SIZE = 100_000

train_recal, test = train_test_split(
    sample, test_size=TEST_SIZE, stratify=sample["y"], random_state=SEED
)
train, recal = train_test_split(
    train_recal, test_size=RECAL_SIZE, stratify=train_recal["y"], random_state=SEED
)
del train_recal

splits = {"train": train, "recalibration": recal, "test": test}
assert [len(df) for df in splits.values()] == [TRAIN_SIZE, RECAL_SIZE, TEST_SIZE]
assert train.index.union(recal.index).union(test.index).size == SAMPLE_SIZE

Is the long-stay rate the same in each set?

In [21]:
for name, df in splits.items():
    print(f"{name}: {len(df):,} stays, {df['y'].mean():.2%} long")

train: 360,000 stays, 23.06% long
recalibration: 40,000 stays, 23.06% long
test: 100,000 stays, 23.06% long


How many test stays will each fairness group have? These are counts only, so
they don't use any test-set outcomes.

In [22]:
for column in ["payment_typology_1", "race"]:
    print(test[column].value_counts(dropna=False).to_string())
    print()

payment_typology_1
Medicare                     48861
Medicaid                     27632
Private Health Insurance     11448
Blue Cross/Blue Shield        7484
Self-Pay                      1304
Managed Care, Unspecified     1168
Federal/State/Local/VA        1027
Miscellaneous/Other            981
Department of Corrections       93
NaN                              2

race
White                     52193
Other Race                26710
Black/African American    19851
Multi-racial               1246



## Step 4: Clean

Everything we look at in this step comes from the training set. The fixes go in
one function, so the recalibration and test sets can get the same fixes,
unchanged, when we use them.

### Labels that differ by year

These are the 17 features the model will use. Which of their levels appear in
only one year?

In [23]:
FEATURES = [
    "age_group",
    "gender",
    "race",
    "ethnicity",
    "zip_code",
    "payment_typology_1",
    "payment_typology_2",
    "payment_typology_3",
    "type_of_admission",
    "emergency_department_indicator",
    "permanent_facility_id",
    "hospital_county",
    "health_service_area",
    "apr_mdc_code",
    "apr_medical_surgical_description",
    "ccsr_diagnosis_code",
    "discharge_year",
]

for column in FEATURES:
    if column == "discharge_year":
        continue
    in_2023 = set(train.loc[train["discharge_year"] == "2023", column].dropna())
    in_2024 = set(train.loc[train["discharge_year"] == "2024", column].dropna())
    if in_2023 != in_2024:
        print(column)
        print("  Only in 2023:", sorted(in_2023 - in_2024))
        print("  Only in 2024:", sorted(in_2024 - in_2023))

age_group
  Only in 2023: ['0 to 17', '18 to 29', '30 to 49', '50 to 69']
  Only in 2024: ['0-17', '18-29', '30-49', '50-69']


permanent_facility_id
  Only in 2023: ['000565', '000598', '000599', '001315']
  Only in 2024: ['000848', '015437', '015485', '015620']
hospital_county
  Only in 2023: ['Manhattan', 'St Lawrence']
  Only in 2024: ['New York', 'Saint Lawrence']


health_service_area
  Only in 2023: ['Capital/Adirond']
  Only in 2024: ['Capital/Adirondacks']


ccsr_diagnosis_code
  Only in 2023: ['FAC007', 'INJ056', 'INJ057', 'INJ070', 'NEO047', 'NEO055', 'PRG015']
  Only in 2024: ['INJ044', 'INJ051', 'INJ053', 'INJ054', 'INJ075', 'MBD023', 'NEO053', 'PNL004', 'PNL008', 'SYM018']


Age, county and service area are the same levels spelled differently, so without
a fix each one would split in two and partly encode the year. We use one spelling
for each.

Facility and diagnosis are codes rather than spellings, so we leave them alone.
Any that are rare get grouped in Step 5.

In [24]:
LABEL_FIXES = {
    "age_group": {
        "0 to 17": "0-17",
        "18 to 29": "18-29",
        "30 to 49": "30-49",
        "50 to 69": "50-69",
    },
    "health_service_area": {"Capital/Adirond": "Capital/Adirondacks"},
    "hospital_county": {"Manhattan": "New York", "St Lawrence": "Saint Lawrence"},
}

### Codes, not descriptions

Three features have a code column and a description column. Does each code have
one description?

In [25]:
CODE_DESCRIPTIONS = {
    "ccsr_diagnosis_code": "ccsr_diagnosis_description",
    "apr_mdc_code": "apr_mdc_description",
    "permanent_facility_id": "facility_name",
}

for code, description in CODE_DESCRIPTIONS.items():
    within_year = train.groupby(["discharge_year", code])[description].nunique()
    across_years = train.groupby(code)[description].unique()
    shared = train.groupby(description)[code].nunique()
    print(code)
    print(f"  Codes: {train[code].nunique()}")
    print(f"  Descriptions: {train[description].nunique()}")
    print(f"  Most descriptions for a code within a year: {within_year.max()}")
    print(
        f"  Most descriptions for a code across years: {across_years.str.len().max()}"
    )
    print(f"  Descriptions used by more than one code: {(shared > 1).sum()}")
    print(f"  Example: {list(across_years[across_years.str.len() > 1].iloc[0])}")

ccsr_diagnosis_code
  Codes: 446
  Descriptions: 875
  Most descriptions for a code within a year: 1
  Most descriptions for a code across years: 2
  Descriptions used by more than one code: 0
  Example: ['Nutritional anemia', 'NUTRITIONAL ANEMIA']
apr_mdc_code
  Codes: 25
  Descriptions: 46
  Most descriptions for a code within a year: 1
  Most descriptions for a code across years: 2
  Descriptions used by more than one code: 0
  Example: ['DISEASES AND DISORDERS OF THE NERVOUS SYSTEM', 'DISEASES & DISORDERS OF THE NERVOUS SYSTEM']
permanent_facility_id
  Codes: 195
  Descriptions: 378
  Most descriptions for a code within a year: 1
  Most descriptions for a code across years: 2
  Descriptions used by more than one code: 4
  Example: ['ALBANY MEDICAL CENTER HOSPITAL', 'Albany Medical Center Hospital']


Within a year, each code has one description. Across years, most codes have two,
and the examples differ only in how they're written. One-hot encoding the
descriptions would split each level by year, so we encode the codes and use the
descriptions only to label results.

A few facility names are used by more than one ID. We keep the IDs as they are,
because year-to-year facility changes are out of scope.

### Blanks

Which features have blanks, and do those stays look different?

In [26]:
blank_counts = train[FEATURES].isna().sum()
BLANK_COLUMNS = blank_counts[blank_counts > 0].index.tolist()

blanks = pd.DataFrame(
    {
        "blank_stays": blank_counts[BLANK_COLUMNS],
        "blank_share": train[BLANK_COLUMNS].isna().mean(),
        "long_rate_if_blank": [
            train.loc[train[c].isna(), "y"].mean() for c in BLANK_COLUMNS
        ],
        "long_rate_if_filled": [
            train.loc[train[c].notna(), "y"].mean() for c in BLANK_COLUMNS
        ],
    }
)
print(blanks.round(3).to_string())

                    blank_stays  blank_share  long_rate_if_blank  long_rate_if_filled
zip_code                   5657        0.016               0.256                0.230
payment_typology_1           19        0.000               0.263                0.231
payment_typology_2       177942        0.494               0.206                0.255
payment_typology_3       310974        0.864               0.224                0.271


Only ZIP and the three payer columns have blanks, and the long-stay rate differs
between blank and filled rows, so a blank tells us something. We fill blanks with
"Unknown" so that each becomes its own level. A one-hot "Unknown" column already
marks these rows, so we don't add a separate missingness indicator.

A blank payer 2 or 3 probably means there's no backup payer, not that a value is
missing. Payer 1 has so few blanks that its "Unknown" level will go into "Other"
in Step 5.

### Apply the fixes

Stop with an error if the cleaned training set doesn't look as expected.

In [27]:
def clean(df):
    """Apply the label fixes and fill blanks. Learns nothing from the data."""
    return df.replace(LABEL_FIXES).fillna({c: "Unknown" for c in BLANK_COLUMNS})


train = clean(train)

levels = train[FEATURES].nunique()
print(levels.to_string())
print(f"Total: {levels.sum()}")

assert train[FEATURES].notna().all().all()
assert levels["age_group"] == 5, levels["age_group"]
assert levels["health_service_area"] == 8, levels["health_service_area"]
assert levels["hospital_county"] == 57, levels["hospital_county"]
assert levels.sum() == 838, levels.sum()

age_group                             5
gender                                3
race                                  4
ethnicity                             4
zip_code                             51
payment_typology_1                   10
payment_typology_2                   10
payment_typology_3                   10
type_of_admission                     3
emergency_department_indicator        2
permanent_facility_id               195
hospital_county                      57
health_service_area                   8
apr_mdc_code                         25
apr_medical_surgical_description      3
ccsr_diagnosis_code                 446
discharge_year                        2
Total: 838


## Step 5: Encode

Every feature is a category with no natural order, even when its levels look like
numbers, so we one-hot encode all 17. Each level gets its own 0/1 column.

### Rare levels

A level with very few stays would get a noisy coefficient. We group any level with
fewer than 100 training stays into "Other". No standard sets this number, so it's
our judgment. We chose it before seeing any model results, and it uses counts, not
outcomes.

How many levels and stays does this affect in each feature?

In [28]:
MIN_STAYS = 100

rare = pd.DataFrame(index=FEATURES, columns=["levels", "rare_levels", "rare_stays"])
for column in FEATURES:
    counts = train[column].value_counts()
    is_rare = counts < MIN_STAYS
    rare.loc[column] = [len(counts), is_rare.sum(), counts[is_rare].sum()]
print(rare.to_string())
print(f"Total rare levels: {rare['rare_levels'].sum()}")

                                 levels rare_levels rare_stays
age_group                             5           0          0
gender                                3           1         26
race                                  4           0          0
ethnicity                             4           0          0
zip_code                             51           0          0
payment_typology_1                   10           1         19
payment_typology_2                   10           1         17
payment_typology_3                   10           2         42
type_of_admission                     3           0          0
emergency_department_indicator        2           0          0
permanent_facility_id               195          21        900
hospital_county                      57           5        325
health_service_area                   8           0          0
apr_mdc_code                         25           1         11
apr_medical_surgical_description      3           1    

Diagnosis and facility are affected most, with many levels but few stays. Where a
feature has one rare level, such as gender "U", that level is just relabeled.

### Fit the encoder

This fits the encoder once on all of train, so we can check what it does. In Step
6, it's refit inside each CV fold, so the validation fold can't affect the grouping.

`drop="first"` leaves out one level from each feature as the reference. Which level
does it leave out, and how many columns do we end up with?

In [29]:
encoder = OneHotEncoder(
    min_frequency=MIN_STAYS, handle_unknown="infrequent_if_exist", drop="first"
)
encoded = encoder.fit_transform(train[FEATURES])

names = pd.Series(encoder.get_feature_names_out())
summary = pd.DataFrame(
    {
        "reference": [
            categories[index]
            for categories, index in zip(encoder.categories_, encoder.drop_idx_)
        ],
        "columns": [names.str.startswith(f"{c}_").sum() for c in FEATURES],
    },
    index=FEATURES,
)
print(summary.to_string())
print()
print(f"Encoded training set: {encoded.shape[0]:,} rows, {encoded.shape[1]} columns")
print(f"Share of values that aren't zero: {encoded.mean():.1%}")

assert encoded.shape == (TRAIN_SIZE, 615), encoded.shape

                                               reference  columns
age_group                                           0-17        4
gender                                                 F        2
race                              Black/African American        3
ethnicity                                   Multi-ethnic        3
zip_code                                             100       50
payment_typology_1                Blue Cross/Blue Shield        9
payment_typology_2                Blue Cross/Blue Shield        9
payment_typology_3                Blue Cross/Blue Shield        8
type_of_admission                              Emergency        2
emergency_department_indicator                         N        1
permanent_facility_id                             000001      174
hospital_county                                   Albany       52
health_service_area                  Capital/Adirondacks        7
apr_mdc_code                                          01       24
apr_medica

The reference is the alphabetically first level that isn't rare. For example, MDC
code "00" is rare, so the reference is "01". We won't interpret these coefficients,
so which level is the reference doesn't matter here.

The result is sparse: each row has at most 17 nonzero values, so it fits easily in
memory even with 615 columns.

### New levels

A validation fold or later set can contain a level that the encoder never saw. What
happens to one?

In [30]:
new_facility = train[FEATURES].iloc[[0]].assign(permanent_facility_id="NEW")
new_zip = train[FEATURES].iloc[[0]].assign(zip_code="NEW")


def active_columns(row, feature):
    """Names of a feature's columns that are 1 for this row."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        values = encoder.transform(row).toarray()[0]
    return [n for n, v in zip(names, values) if v and n.startswith(f"{feature}_")]


print("New facility:", active_columns(new_facility, "permanent_facility_id"))
print("New ZIP:", active_columns(new_zip, "zip_code"))

New facility: ['permanent_facility_id_infrequent_sklearn']
New ZIP: []


A new facility goes to "Other", which scikit-learn labels `infrequent_sklearn`. A new
ZIP gets no column at all, because ZIP has no rare levels and so no "Other". All
zeros means the model treats that stay like the reference level.

That second case affects only features with no rare levels, which have many stays at
every level, so a new level there is unlikely. We'll check for it when we use the
other sets.